In [ ]:
# Sanity check: list all RAW files for the eclipse shoot and decode + display
# just the first one, to confirm rawpy can read these Canon .CR3 files at all
# before doing anything more elaborate.
import glob
import rawpy
import matplotlib.pyplot as plt

raw_dir = "/mnt/c/Users/crueh/Pictures/Raw/2026_08_12"
raw_files = sorted(glob.glob(f"{raw_dir}/*.CR3"))

first_raw = raw_files[0]
print(first_raw)

with rawpy.imread(first_raw) as raw:
    rgb = raw.postprocess()  # default settings: auto white balance + auto brightness

plt.figure(figsize=(10, 8))
plt.imshow(rgb)
plt.axis("off")
plt.show()

In [ ]:
# First slideshow: decode every RAW at half resolution (much faster than full
# size, plenty sharp for a scrubbing preview) and build a self-contained
# HTML/JS slider that swaps between base64-embedded thumbnails.
#
# We use plain HTML/JS here instead of ipywidgets' interact() because
# ipywidgets needs a live "widget manager" extension in the notebook
# frontend to render as an interactive slider -- that wasn't available in
# this environment, so interact() silently fell back to a static display.
# A self-contained <input type="range"> has no such dependency.
import base64
from io import BytesIO
from PIL import Image
from IPython.display import HTML

thumbs = []
for i, path in enumerate(raw_files):
    with rawpy.imread(path) as raw:
        img = raw.postprocess(half_size=True)
    im = Image.fromarray(img)
    im.thumbnail((480, 480))
    buf = BytesIO()
    im.save(buf, format="JPEG", quality=70)
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    thumbs.append(f"data:image/jpeg;base64,{b64}")
    print(f"\r{i + 1}/{len(raw_files)}", end="")

frames_js = ",".join(f'"{t}"' for t in thumbs)
names_js = ",".join(f'"{path.split("/")[-1]}"' for path in raw_files)

html = f"""
<div>
  <img id="slideshow-img" src="{thumbs[0]}" style="max-width:480px; display:block;">
  <input id="slideshow-range" type="range" min="0" max="{len(thumbs) - 1}" value="0" style="width:480px;">
  <div id="slideshow-label"></div>
</div>
<script>
(function() {{
  const frames = [{frames_js}];
  const names = [{names_js}];
  const img = document.getElementById("slideshow-img");
  const range = document.getElementById("slideshow-range");
  const label = document.getElementById("slideshow-label");
  function update(i) {{
    img.src = frames[i];
    label.textContent = (parseInt(i) + 1) + " / " + frames.length + " — " + names[i];
  }}
  range.addEventListener("input", (e) => update(e.target.value));
  update(0);
}})();
</script>
"""
HTML(html)

In [ ]:
# Check whether exposure settings (ISO/shutter) changed mid-shoot -- if so,
# frames won't be directly comparable in brightness and need correcting.
# rawpy doesn't expose EXIF the way exifread/exiftool would (and neither of
# those actually parses Canon's .CR3 container), but raw.other pulls the
# same info straight from libraw.
exposure_log = []
for path in raw_files:
    with rawpy.imread(path) as raw:
        o = raw.other
    exposure_log.append({
        "file": path.split("/")[-1],
        "iso": o.iso_speed,
        "shutter": o.shutter_speed,
        "time": o.timestamp,
    })

print(f"{'#':>3} {'file':14} {'ISO':>6} {'shutter':>10}   time")
prev_iso, prev_shutter = None, None
for i, e in enumerate(exposure_log):
    changed = (e["iso"] != prev_iso) or (e["shutter"] != prev_shutter)
    marker = " <-- changed" if changed and i > 0 else ""
    print(f"{i + 1:3d} {e['file']:14} {e['iso']:6.0f} {e['shutter']:10.5f}   {e['time']}{marker}")
    prev_iso, prev_shutter = e["iso"], e["shutter"]

# Result: yes -- shutter speed changes 3 times during the partial-eclipse
# phase (1/2000s -> 1/40s -> 1/100s, all ISO 400) as the crescent thinned,
# then ISO/shutter jump around a lot more near the end as the sun set. This
# is why the brightness-normalization step below is needed.

## Aligning the sun and correcting exposure

The telescope wasn't tracking perfectly, so the sun drifts around the frame
over the ~2.5 hour sequence. This cell aligns every frame so the sun stays
fixed and the moon appears to move across it instead, and also fixes a
brightness problem: `rawpy`'s default `postprocess()` applies its own
automatic brightness boost per frame, which caused visible flicker across
the sequence (each frame gets brightened by a different, content-dependent
amount). Here we decode with `no_auto_bright=True` instead and handle
brightness matching explicitly and consistently in the next cell.

**Alignment approach:** fit a circle to the sun's visible limb per frame
(via the convex hull of its thresholded contour, which discards the
concave notch where the moon overlaps), holding the radius fixed at a
value measured from the widest, most reliable frames -- a free 3-parameter
fit (center + radius) is ill-conditioned on the thin arcs near totality,
but a 2-parameter fit (center only, radius known) stays stable. For frames
where even that isn't reliable (arc too short or too sparsely sampled),
the position is interpolated in time from neighboring confident frames
instead of trusting a shaky fit.

In [ ]:
import numpy as np
import cv2

def decode_asshot(path):
    """No auto-brightness (that caused artifacts) and no metadata-based
    exposure guess either -- just the camera's own chosen exposure, demosaiced
    linearly and given a single fixed display gamma ourselves.

    output_bps=16 + gamma=(1,1) gets us a linear-light image (proportional
    to actual sensor exposure, no tone curve baked in yet); we then apply
    one consistent sRGB-ish gamma (1/2.2) ourselves before dropping to 8-bit,
    so every frame goes through the exact same brightness mapping.
    """
    with rawpy.imread(path) as raw:
        linear = raw.postprocess(
            half_size=True,
            no_auto_bright=True,
            use_camera_wb=True,
            output_bps=16,
            gamma=(1, 1),
        ).astype(np.float64)
    srgb = np.clip(linear / 65535.0, 0, 1) ** (1 / 2.2)
    return (srgb * 255).astype(np.uint8)

frames = []
for i, path in enumerate(raw_files):
    frames.append(decode_asshot(path))
    print(f"\r{i + 1}/{len(raw_files)}", end="")
print(f"\ndecoded {len(frames)} frames (no auto-bright, no forced exposure match yet)")

# --- geometry: locate the sun in each frame ---

def sun_mask(gray):
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return mask

def get_hull_points(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    largest = max(contours, key=cv2.contourArea)
    return cv2.convexHull(largest).reshape(-1, 2).astype(np.float64)

def fit_circle(points):
    x, y = points[:, 0], points[:, 1]
    A = np.column_stack([2 * x, 2 * y, np.ones_like(x)])
    b = x**2 + y**2
    sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    cx, cy, c = sol
    return cx, cy, np.sqrt(c + cx**2 + cy**2)

def fit_center_fixed_radius(points, r, cx0, cy0, iters=15):
    cx, cy = cx0, cy0
    for _ in range(iters):
        dx, dy = points[:, 0] - cx, points[:, 1] - cy
        d = np.maximum(np.sqrt(dx**2 + dy**2), 1e-6)
        e = d - r
        J = np.column_stack([-dx / d, -dy / d])
        try:
            delta = np.linalg.solve(J.T @ J, -(J.T @ e))
        except np.linalg.LinAlgError:
            break
        cx, cy = cx + delta[0], cy + delta[1]
        if np.linalg.norm(delta) < 1e-4:
            break
    return cx, cy

def hull_angular_span_deg(points, cx, cy):
    angles = np.sort(np.arctan2(points[:, 1] - cy, points[:, 0] - cx))
    gaps = np.diff(angles)
    wrap_gap = angles[0] + 2 * np.pi - angles[-1]
    return np.degrees(2 * np.pi - max(gaps.max() if len(gaps) else 0, wrap_gap))

masks, hull_points_list, circles = [], [], []
for img in frames:
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = sun_mask(gray)
    pts = get_hull_points(mask)
    masks.append(mask)  # kept around for the brightness measurement below
    hull_points_list.append(pts)
    circles.append(fit_circle(pts))
circles = np.array(circles)
hull_counts = np.array([len(p) for p in hull_points_list])

well_conditioned = hull_counts >= np.percentile(hull_counts, 50)
r_global = np.median(circles[well_conditioned, 2])

refit = np.array([
    fit_center_fixed_radius(pts, r_global, cx0, cy0)
    for pts, (cx0, cy0, _) in zip(hull_points_list, circles)
])
spans = np.array([hull_angular_span_deg(pts, cx, cy) for pts, (cx, cy) in zip(hull_points_list, refit)])

MIN_SPAN_DEG, MIN_HULL_PTS = 120, 20
confident = (spans >= MIN_SPAN_DEG) & (hull_counts >= MIN_HULL_PTS)
idx = np.arange(len(frames))
final_cx = np.interp(idx, idx[confident], refit[confident, 0])
final_cy = np.interp(idx, idx[confident], refit[confident, 1])
print(f"{confident.sum()}/{len(frames)} frames trusted directly, {(~confident).sum()} interpolated")

# --- brightness: the sun's actual surface brightness doesn't change as the
# moon covers it (only the exposed *area* shrinks) -- so the mean brightness
# of the illuminated pixels is a direct, measured stand-in for "how this
# frame's exposure compares," no metadata guessing needed.
#
# (We initially tried computing a correction straight from the ISO/shutter
# ratio in exposure_log, but that was wrong: the photographer deliberately
# lengthened the shutter as the crescent thinned, precisely to keep it
# correctly exposed. Dividing that back out just re-darkened the frames
# that were already correctly compensated. Measuring actual pixel
# brightness sidesteps that entirely.) ---
sun_brightness = np.array([
    cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)[mask > 0].mean()
    for img, mask in zip(frames, masks)
])
target_brightness = np.median(sun_brightness[confident])
raw_scale = target_brightness / np.maximum(sun_brightness, 1.0)
# Interpolate the scale factor too, for the same low-confidence frames as above.
brightness_scale = np.interp(idx, idx[confident], raw_scale[confident])
print(f"target sun brightness: {target_brightness:.1f}/255, scale range: {brightness_scale.min():.2f}x-{brightness_scale.max():.2f}x")

target_cx, target_cy = np.median(final_cx), np.median(final_cy)
crop_r = int(r_global * 1.3)

aligned = []
for img, cx, cy, scale in zip(frames, final_cx, final_cy, brightness_scale):
    corrected = np.clip(img.astype(np.float64) * scale, 0, 255).astype(np.uint8)
    dx, dy = target_cx - cx, target_cy - cy
    M = np.array([[1, 0, dx], [0, 1, dy]], dtype=np.float64)
    shifted = cv2.warpAffine(corrected, M, (img.shape[1], img.shape[0]))
    x0, x1 = int(target_cx - crop_r), int(target_cx + crop_r)
    y0, y1 = int(target_cy - crop_r), int(target_cy + crop_r)
    pad_x0, pad_y0 = max(0, -x0), max(0, -y0)
    pad_x1, pad_y1 = max(0, x1 - img.shape[1]), max(0, y1 - img.shape[0])
    cropped = shifted[max(y0, 0):y1, max(x0, 0):x1]
    if pad_x0 or pad_y0 or pad_x1 or pad_y1:
        cropped = np.pad(cropped, ((pad_y0, pad_y1), (pad_x0, pad_x1), (0, 0)))
    aligned.append(cropped)

# Slides 56-73 (1-indexed): telescope covered, force pure black.
for i in range(55, 73):
    aligned[i] = np.zeros_like(aligned[i])

print(f"re-aligned {len(aligned)} frames, crop size: {aligned[0].shape[1]}x{aligned[0].shape[0]}")

In [ ]:
# The brightness-matching step above only makes frames CONSISTENT with each
# other (median ~96/255) -- it doesn't make the sun look bright. Push
# everything up with one single, uniform multiplier (same factor for every
# frame, not content-based per-frame) so it stays flicker-free while making
# the disc read close to white, as the actual sun should look.
TARGET_DISPLAY = 240  # near-white, small headroom left so the disc doesn't fully flatten out
boost = TARGET_DISPLAY / target_brightness
brightness_scale_boosted = brightness_scale * boost
print(f"boosting by {boost:.2f}x so the sun reads ~{TARGET_DISPLAY}/255 (was {target_brightness:.1f}/255)")

aligned = []
for img, cx, cy, scale in zip(frames, final_cx, final_cy, brightness_scale_boosted):
    corrected = np.clip(img.astype(np.float64) * scale, 0, 255).astype(np.uint8)
    dx, dy = target_cx - cx, target_cy - cy
    M = np.array([[1, 0, dx], [0, 1, dy]], dtype=np.float64)
    shifted = cv2.warpAffine(corrected, M, (img.shape[1], img.shape[0]))
    x0, x1 = int(target_cx - crop_r), int(target_cx + crop_r)
    y0, y1 = int(target_cy - crop_r), int(target_cy + crop_r)
    pad_x0, pad_y0 = max(0, -x0), max(0, -y0)
    pad_x1, pad_y1 = max(0, x1 - img.shape[1]), max(0, y1 - img.shape[0])
    cropped = shifted[max(y0, 0):y1, max(x0, 0):x1]
    if pad_x0 or pad_y0 or pad_x1 or pad_y1:
        cropped = np.pad(cropped, ((pad_y0, pad_y1), (pad_x0, pad_x1), (0, 0)))
    aligned.append(cropped)

# Slides 56-73 (1-indexed): telescope covered, force pure black.
for i in range(55, 73):
    aligned[i] = np.zeros_like(aligned[i])

print(f"re-aligned {len(aligned)} frames with boosted brightness, crop size: {aligned[0].shape[1]}x{aligned[0].shape[0]}")

In [ ]:
# Same HTML/JS slider approach as the very first slideshow, but now showing
# the final aligned + exposure-corrected frames -- this is the "pac-man"
# eclipse movie: sun fixed in place, moon visibly moving across it.
import base64
from io import BytesIO
from PIL import Image
from IPython.display import HTML

aligned_thumbs = []
for i, img in enumerate(aligned):
    im = Image.fromarray(img)
    buf = BytesIO()
    im.save(buf, format="JPEG", quality=80)
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    aligned_thumbs.append(f"data:image/jpeg;base64,{b64}")

frames_js = ",".join(f'"{t}"' for t in aligned_thumbs)
names_js = ",".join(f'"{path.split("/")[-1]}"' for path in raw_files)

html = f"""
<div>
  <img id="aligned-slideshow-img" src="{aligned_thumbs[0]}" style="max-width:480px; display:block;">
  <input id="aligned-slideshow-range" type="range" min="0" max="{len(aligned_thumbs) - 1}" value="0" style="width:480px;">
  <div id="aligned-slideshow-label"></div>
</div>
<script>
(function() {{
  const frames = [{frames_js}];
  const names = [{names_js}];
  const img = document.getElementById("aligned-slideshow-img");
  const range = document.getElementById("aligned-slideshow-range");
  const label = document.getElementById("aligned-slideshow-label");
  function update(i) {{
    img.src = frames[i];
    label.textContent = (parseInt(i) + 1) + " / " + frames.length + " — " + names[i];
  }}
  range.addEventListener("input", (e) => update(e.target.value));
  update(0);
}})();
</script>
"""
HTML(html)

## Diagnostics

Scratch cells used to pick the `MIN_SPAN_DEG` / `MIN_HULL_PTS` thresholds
used in the confidence check above, kept here for reference.

In [ ]:
# The 15 frames with the smallest detected arc -- worth checking these
# aren't just noise (e.g. from the covered-telescope stretch) before
# trusting the span/point-count numbers to set thresholds.
order = np.argsort(spans)
for i in order[:15]:
    print(f"frame {i:3d} ({raw_files[i].split('/')[-1]}): span={spans[i]:6.1f}deg  hull_pts={len(hull_points_list[i])}")

In [ ]:
# Sweep a few (span, point-count) threshold combos to see how many frames
# end up "confident" under each -- landed on span>=120 & pts>=20 as a
# reasonable middle ground (neither trusting too-thin arcs nor discarding
# perfectly good frames).
for min_span, min_pts in [(60, 0), (90, 0), (90, 15), (120, 20), (150, 25)]:
    c = (spans >= min_span) & (hull_counts >= min_pts)
    print(f"span>={min_span:3d} & pts>={min_pts:2d}: {c.sum():3d}/{len(frames)} confident")

## Exporting the results

Two shareable outputs from the aligned/corrected `aligned` frames: a small
animated GIF, and a full sensor-resolution movie (which needs a fresh RAW
decode, since `aligned` above was only ever decoded at half resolution for
speed).

In [ ]:
# Animated GIF, downscaled + palette-quantized to keep the file to a few MB.
# GIF has no real inter-frame compression (each frame is basically an
# independent indexed-color image), so total size scales almost linearly
# with resolution x frame count x color count -- there's no trick to shrink
# it further besides reducing one of those.
import os
from PIL import Image

GIF_PATH = "eclipse.gif"
GIF_MAX_SIDE = 220   # GIF has no real inter-frame compression -- size scales with pixel count
GIF_COLORS = 128
GIF_FRAME_MS = 80

def to_pil(arr, max_side, colors):
    im = Image.fromarray(arr)
    im.thumbnail((max_side, max_side))
    return im.quantize(colors=colors, method=Image.MEDIANCUT, dither=Image.NONE)

gif_frames = [to_pil(f, GIF_MAX_SIDE, GIF_COLORS) for f in aligned]
gif_frames[0].save(
    GIF_PATH,
    save_all=True,
    append_images=gif_frames[1:],
    duration=GIF_FRAME_MS,
    loop=0,
    optimize=True,
)

size_mb = os.path.getsize(GIF_PATH) / 1e6
print(f"wrote {GIF_PATH}: {gif_frames[0].size[0]}x{gif_frames[0].size[1]}, {len(gif_frames)} frames, {size_mb:.2f} MB")

In [ ]:
# Full sensor-resolution movie. Re-decodes every RAW from scratch at full
# size (half_size=False) -- necessary because holding all 120 full-res
# frames in memory at once (~8GB+) isn't practical, so instead of building
# an `aligned`-style list first, each frame is decoded, corrected, cropped
# and immediately streamed into the video writer, one at a time.
#
# Reuses the position/brightness values (final_cx, final_cy, crop_r,
# brightness_scale_boosted) computed above at half resolution -- those are
# just scaled by 2x for the coordinates (half_size=True decodes at exactly
# 1/2 linear resolution) since the brightness scale factor itself doesn't
# depend on resolution.
import imageio.v2 as imageio

MOVIE_PATH = "eclipse_full_res.mp4"
FPS = 12
FULL_RES_SCALE = 2  # half_size=True decoded at 1/2 linear resolution -- undo that here

full_cx = final_cx * FULL_RES_SCALE
full_cy = final_cy * FULL_RES_SCALE
full_target_cx, full_target_cy = target_cx * FULL_RES_SCALE, target_cy * FULL_RES_SCALE
full_crop_r = crop_r * FULL_RES_SCALE
full_out_size = full_crop_r * 2

writer = imageio.get_writer(MOVIE_PATH, fps=FPS, codec="libx264", quality=8, macro_block_size=None)

for i, path in enumerate(raw_files):
    if 55 <= i <= 72:  # telescope covered -- skip the decode entirely, write black
        frame = np.zeros((full_out_size, full_out_size, 3), dtype=np.uint8)
    else:
        with rawpy.imread(path) as raw:
            linear = raw.postprocess(
                half_size=False,
                no_auto_bright=True,
                use_camera_wb=True,
                output_bps=16,
                gamma=(1, 1),
            ).astype(np.float64)
        srgb = np.clip(linear / 65535.0, 0, 1) ** (1 / 2.2)
        img = (srgb * 255).astype(np.uint8)

        corrected = np.clip(img.astype(np.float64) * brightness_scale_boosted[i], 0, 255).astype(np.uint8)
        dx, dy = full_target_cx - full_cx[i], full_target_cy - full_cy[i]
        M = np.array([[1, 0, dx], [0, 1, dy]], dtype=np.float64)
        shifted = cv2.warpAffine(corrected, M, (img.shape[1], img.shape[0]))

        x0, x1 = int(full_target_cx - full_crop_r), int(full_target_cx + full_crop_r)
        y0, y1 = int(full_target_cy - full_crop_r), int(full_target_cy + full_crop_r)
        pad_x0, pad_y0 = max(0, -x0), max(0, -y0)
        pad_x1, pad_y1 = max(0, x1 - img.shape[1]), max(0, y1 - img.shape[0])
        frame = shifted[max(y0, 0):y1, max(x0, 0):x1]
        if pad_x0 or pad_y0 or pad_x1 or pad_y1:
            frame = np.pad(frame, ((pad_y0, pad_y1), (pad_x0, pad_x1), (0, 0)))

    writer.append_data(frame)
    print(f"\r{i + 1}/{len(raw_files)}", end="")

writer.close()
size_mb = os.path.getsize(MOVIE_PATH) / 1e6
print(f"\nwrote {MOVIE_PATH}: {full_out_size}x{full_out_size}, {len(raw_files)} frames @ {FPS}fps, {size_mb:.1f} MB")

# Note: len(raw_files) in the message above is the number of files that
# EXIST on disk at the time this cell runs, not a hardcoded 120 -- if you
# delete trailing/unwanted RAW files (as happened here, trimming the
# sun-below-horizon tail) before running this cell, that's exactly how many
# frames the movie will have.

In [ ]:
# The movie above is much larger than it needs to be: imageio's "quality"
# 0-10 knob for libx264 maps to a very high, non-obvious bitrate. Re-encode
# with an explicit CRF instead for a much better size/quality tradeoff, at
# the same resolution -- this is a pure video-to-video transcode (ffmpeg
# re-reads the already-rendered mp4), so it's fast and doesn't touch the
# RAW files at all.
import subprocess
import imageio_ffmpeg

COMPRESSED_PATH = "eclipse_full_res_compressed.mp4"
CRF = 23  # lower = higher quality/bigger file; 23 is visually near-lossless for x264

ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
subprocess.run([
    ffmpeg, "-y", "-i", MOVIE_PATH,
    "-c:v", "libx264", "-preset", "slow", "-crf", str(CRF), "-pix_fmt", "yuv420p",
    COMPRESSED_PATH,
    "-loglevel", "error",
], check=True)

before_mb = os.path.getsize(MOVIE_PATH) / 1e6
after_mb = os.path.getsize(COMPRESSED_PATH) / 1e6
print(f"{MOVIE_PATH}: {before_mb:.1f} MB -> {COMPRESSED_PATH}: {after_mb:.1f} MB (CRF {CRF})")